# Example for reading PhotoCalib




- refer to : https://dp1.lsst.io/tutorials/notebook/104/notebook-104-1.html
---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-29
- **Last update:** 2026-06-29


## Imports

In [ ]:
import gc
import logging
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees, Point2D

In [ ]:
from libExtractLightcurves import safe_name, find_col, dataset_type_exists

## Logging

In [ ]:
# 1. Récupérer le logger racine (ou créez un logger spécifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Définir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Éviter la duplication des handlers si la cellule est exécutée plusieurs fois
if not log.handlers:
    # 4. Créer un handler qui écrit vers la sortie standard (capturée par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Définir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour vérifier que ça fonctionne
log.info("Le logging est configuré et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
!ls data_MergeVisits_02_out

In [ ]:
# ── Output directories ───────────────────────────────────────────────────

DIR_DATA_IN = "data_MergeVisits_02_out"
filename_input = "all_stars_lightcurves_mjd.csv"
log.info(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
log.info(f"File input : {filename_input}")

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"
BANDSEL = "r"

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
log.debug(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")


log.info("Butler configuration done.")

## Load Light Curves and Visits

In [ ]:
fullfilename = os.path.join(DIR_DATA_IN, filename_input)
df = pd.read_csv(fullfilename)
display(df.head())
log.info("Loaded %d entries from %s", len(df), filename_input)

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised | repo: {repo}")

In [ ]:
# List all source-table-related types actually in the registry
all_pvi_types = [
    d.name for d in registry.queryDatasetTypes() if "visit" in d.name.lower() and "image" in d.name.lower()
]
print("img--related dataset types in registry:")
for t in sorted(all_pvi_types):
    print(f"  {t}")

In [ ]:
# Prioritised list of candidate object-table dataset type names
PVI_TABLE_CANDIDATES = [
    "preliminary_visit_image",
    "legacy_visit_image",
    "visit_image",
]


# Pick the first candidate that is registered
PVI_DATASET = None
for name in PVI_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        PVI_DATASET = name
        log.info(f"\n✔ Selected PVI dataset type: '{PVI_DATASET}'")
        break

if PVI_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

## Inspect the schema of the PVI (probe on first target)

We load a single PVI from a representative visit to check which
informations are available

In [ ]:
# Use the coordinates of the first target to locate a representative tract
first_row = df.iloc[0]

target_name = first_row["simbad_id"]
visit_id = first_row["visit"]
ra = first_row["src_ra"]
dec = first_row["src_dec"]
x = first_row["x"]
y = first_row["y"]
pos = Point2D(x, y)
det = first_row["detector"]

In [ ]:
msg = f"first visit {visit_id} for target {target_name} tested"
log.info(msg)
msg = f"(ra,dec) = ({ra:.2f},{dec:.2f}) \t (x,y) = ({x:.1f},{y:.1f}) \t det {det}"
log.info(msg)

In [ ]:
band = BANDSEL

In [ ]:
# Build timespan
t1 = Time(MJD_START, format="mjd", scale="tai")
t2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(t1, t2)
log.info("Timespan MJD [%.1f, %.1f]  (delta=%.0f days)", t1.mjd, t2.mjd, t2.mjd - t1.mjd)

- Below I get all references for the visit that match this source. 

In [ ]:
dataset_refs = butler.query_datasets(
    PVI_DATASET,
    where="band.name = :band AND \
                                     visit.timespan OVERLAPS :timespan AND \
                                     visit_detector_region.region OVERLAPS POINT(:ra, :dec)",
    bind={"band": band, "timespan": timespan, "ra": ra, "dec": dec},
    order_by=["visit.timespan.begin"],
)

In [ ]:
len(dataset_refs)

- load the Processed image

In [ ]:
pvi = butler.get(dataset_refs[0])

- Access to the needed information

In [ ]:
# info on visit
info = pvi.getInfo()
# info on calibration
photocal = pvi.getPhotoCalib()
# PSF
psf = pvi.getPsf()
# info on CCD
det = pvi.getDetector()
# info on CCD calibrations
meta = pvi.getMetadata()

In [ ]:
info.getVisitInfo()

In [ ]:
# another visitid , not the good one
visit_count = info.id
visit_count

In [ ]:
# here I access to the Visit number
visitid = pvi.visitInfo.id
visitid

In [ ]:
# need the detector name
det.getName()

In [ ]:
# need the detector number
det.getId()

In [ ]:
# keep the mean calibration over CCD
photocal.getCalibrationMean()

In [ ]:
# keep its error over the mean
photocal.getCalibrationErr()

In [ ]:
# need the local calibration fa ctor
photocal.getLocalCalibration(pos)

In [ ]:
# Need the zero point
2.5 * np.log10(photocal.getInstFluxAtZeroMagnitude())